# Jacobian determinant in the density push-forward formula

This notebook generates `fig:monge-jacobian-pushforward-density`.  It illustrates the change-of-variables formula

$$
\rho_\beta(T(x)) = \frac{\rho_\alpha(x)}{|\det DT(x)|},
$$

for a smooth diffeomorphism.  The source density is uniform, so any nonuniformity in the target density is entirely caused by the Jacobian determinant.  The map compresses area near the center of the square and then applies a small rotation; the target density therefore forms a visible central bump where the grid cells are smallest.

In [1]:
from pathlib import Path
import shutil
import sys

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import LinearSegmentedColormap, to_rgb
from PIL import Image
import subprocess
import tempfile

ROOT = Path.cwd()
if not (ROOT / "notebooks-figures").exists():
    ROOT = ROOT.parent
sys.path.append(str(ROOT / "notebooks-figures"))

from figure_style import BLUE, RED, VIOLET, interp_color, figure_dir, remove_axes, save_pdf, setup_matplotlib

setup_matplotlib()

NAME = "monge-jacobian-pushforward-density"
OUT = figure_dir(NAME)
ARXIV_OUT = ROOT / "arxiv" / "figures"
THUMB_OUT = ROOT / "notebooks-figures" / "thumbnails"
ARXIV_OUT.mkdir(parents=True, exist_ok=True)
THUMB_OUT.mkdir(parents=True, exist_ok=True)


## A simple diffeomorphism with analytic determinant

We use a separable compression followed by a small rigid rotation.  Let

$$
\nu(x)=x+\frac{a}{2\pi}\sin(2\pi x),\qquad
\nu'(x)=1+a\cos(2\pi x)>0.
$$

For $a>0$, the derivative is smallest at $x=1/2$, so the map compresses the middle of the interval.  We apply this in both coordinates and then rotate by a small angle $\theta$ around the center:

$$
T(x,y)=\frac12\mathbf 1 + R_\theta\begin{pmatrix}\nu(x)-1/2\\ \nu(y)-1/2\end{pmatrix}.
$$

Since the rotation has determinant one, the Jacobian determinant is simply

$$
\det DT(x,y)=\nu'(x)\nu'(y),
$$

which is smallest near the center of the source square.  This makes the pushed-forward density bump-like and keeps the geometry easy to read.

In [2]:
a = 0.68
theta = np.deg2rad(10.0)
c, s = np.cos(theta), np.sin(theta)
R = np.array([[c, -s], [s, c]])
RT = R.T
CENTER = np.array([0.5, 0.5])


def nu(x):
    return x + a * np.sin(2 * np.pi * x) / (2 * np.pi)


def nup(x):
    return 1 + a * np.cos(2 * np.pi * x)


def Txy(x, y):
    u = nu(x) - 0.5
    v = nu(y) - 0.5
    return 0.5 + c * u - s * v, 0.5 + s * u + c * v


def det_jacobian(x, y):
    return nup(x) * nup(y)

# Sanity check on a fine grid: the map is orientation preserving.
grid = np.linspace(0, 1, 501)
X, Y = np.meshgrid(grid, grid, indexing="xy")
J = det_jacobian(X, Y)
print("determinant range", float(J.min()), float(J.max()))
assert J.min() > 0

# Finite-difference check of the analytic determinant.
rng = np.random.default_rng(0)
pts = rng.uniform(0.05, 0.95, (20, 2))
h = 1e-6
fd_errors = []
for x, y in pts:
    txp = np.array(Txy(x + h, y))
    txm = np.array(Txy(x - h, y))
    typ = np.array(Txy(x, y + h))
    tym = np.array(Txy(x, y - h))
    Jfd = np.column_stack(((txp - txm) / (2 * h), (typ - tym) / (2 * h)))
    fd_errors.append(abs(np.linalg.det(Jfd) - det_jacobian(x, y)))
print("max finite-difference determinant error", float(max(fd_errors)))

determinant range 0.10239999999999996 2.8224000000000005
max finite-difference determinant error 1.2846168573332761e-10


## Inverting the map on a display grid

The inverse is explicit up to the scalar inverse of $\nu$.  We undo the rotation, check whether the point belongs to the rotated square, and then invert $\nu$ by one-dimensional interpolation in each coordinate.  The target density is evaluated from the determinant formula.

In [3]:
def inverse_nu(z):
    x_grid = np.linspace(0, 1, 8001)
    u_grid = nu(x_grid)
    return np.interp(np.ravel(z), u_grid, x_grid).reshape(np.shape(z))


def inverse_map(Xt, Yt):
    q0 = Xt - 0.5
    q1 = Yt - 0.5
    u = RT[0, 0] * q0 + RT[0, 1] * q1 + 0.5
    v = RT[1, 0] * q0 + RT[1, 1] * q1 + 0.5
    inside = (u >= 0) & (u <= 1) & (v >= 0) & (v <= 1)
    x = inverse_nu(np.clip(u, 0, 1))
    y = inverse_nu(np.clip(v, 0, 1))
    return x, y, inside

# Bounding box of the rotated square, with a small margin for the frame.
corners = np.array([[0, 0], [1, 0], [1, 1], [0, 1]])
Tc = np.column_stack(Txy(corners[:, 0], corners[:, 1]))
margin = 0.055
xmin, ymin = Tc.min(axis=0) - margin
xmax, ymax = Tc.max(axis=0) + margin

N = 430
x_plot = np.linspace(xmin, xmax, N)
y_plot = np.linspace(ymin, ymax, N)
Xt, Yt = np.meshgrid(x_plot, y_plot, indexing="xy")
Xi, Yi, inside = inverse_map(Xt, Yt)
rho_raw = 1.0 / det_jacobian(Xi, Yi)
rho_beta = np.where(inside, rho_raw, np.nan)
dx = (xmax - xmin) / (N - 1)
dy = (ymax - ymin) / (N - 1)
mass_check = np.nansum(np.where(inside, rho_raw, 0.0)) * dx * dy
print(
    "target density range",
    float(np.nanmin(rho_beta)),
    float(np.nanmax(rho_beta)),
    "quadrature mass",
    float(mass_check),
)

target density range 0.35431371635718345 9.748176164299663 quadrature mass 1.000062840472698


## Draw the source grid and the pushed-forward density

The source panel shows a uniform density on the unit square with a colored grid.  The target panel overlays the deformed grid on the pushed-forward density.  The darkest region appears at the center, exactly where the grid cells have been compressed by the map.

In [4]:
import matplotlib.patheffects as pe


def blend(color, amount=0.55):
    rgb = np.array(to_rgb(color))
    return tuple((1 - amount) * rgb + amount * np.ones(3))


def draw_frame(ax, points, lw=0.80):
    pts = np.vstack([points, points[0]])
    ax.plot(pts[:, 0], pts[:, 1], color="#242424", lw=lw, zorder=10)


def grid_color(t):
    if t <= 0.5:
        return interp_color(t / 0.5, RED, VIOLET)
    return interp_color((t - 0.5) / 0.5, VIOLET, BLUE)


def draw_grid(ax, transform=None, n=15, samples=360, alpha=0.88, lw=0.60):
    vals = np.linspace(0, 1, n)
    r = np.linspace(0, 1, samples)
    for v in vals:
        col = grid_color(v)
        x = np.full_like(r, v)
        y = r.copy()
        if transform is not None:
            x, y = transform(x, y)
        ax.plot(x, y, color=col, lw=lw, alpha=alpha, zorder=6)
    for v in vals:
        col = blend(grid_color(v), 0.42)
        x = r.copy()
        y = np.full_like(r, v)
        if transform is not None:
            x, y = transform(x, y)
        ax.plot(x, y, color=col, lw=0.50 * lw / 0.60, alpha=0.72 * alpha, zorder=6)


def draw_source_panel(path):
    fig, ax = plt.subplots(figsize=(2.55, 2.55))
    remove_axes(ax)
    ax.set_aspect("equal")
    ax.set_xlim(-0.018, 1.018)
    ax.set_ylim(-0.018, 1.018)
    ax.add_patch(plt.Rectangle((0, 0), 1, 1, facecolor=blend(RED, 0.90), edgecolor="none", zorder=0))
    draw_grid(ax, transform=None, n=15, samples=240, alpha=0.92, lw=0.62)
    draw_frame(ax, np.array([[0, 0], [1, 0], [1, 1], [0, 1]]))
    save_pdf(fig, path, pad_inches=0.016)
    plt.close(fig)


def draw_target_panel(path):
    fig, ax = plt.subplots(figsize=(2.55, 2.55))
    remove_axes(ax)
    ax.set_aspect("equal")
    ax.set_xlim(xmin, xmax)
    ax.set_ylim(ymin, ymax)

    cmap = LinearSegmentedColormap.from_list(
        "density_blue",
        ["#ffffff", "#eef4fb", "#bdd2ea", "#729bc8", BLUE],
    )
    vmax = np.nanquantile(rho_beta, 0.975)
    rgba = cmap(np.clip(np.nan_to_num(rho_beta, nan=0.0), 0, vmax) / vmax)
    rgba[..., 3] = inside.astype(float)
    ax.imshow(
        rgba,
        extent=[xmin, xmax, ymin, ymax],
        origin="lower",
        interpolation="bicubic",
        zorder=0,
    )
    draw_grid(ax, transform=Txy, n=15, samples=520, alpha=0.72, lw=0.54)
    levels = np.nanquantile(rho_beta, np.linspace(0.46, 0.94, 7))
    contours = ax.contour(
        Xt,
        Yt,
        rho_beta,
        levels=np.unique(levels),
        colors="#1f2d3a",
        linewidths=0.48,
        alpha=0.66,
        zorder=8,
    )
    contours.set_path_effects([pe.Stroke(linewidth=0.82, foreground="white", alpha=0.32), pe.Normal()])
    draw_frame(ax, Tc)
    save_pdf(fig, path, pad_inches=0.016)
    plt.close(fig)


def copy_to_arxiv(pdf_name):
    shutil.copyfile(OUT / pdf_name, ARXIV_OUT / f"{NAME}--{pdf_name}")


draw_source_panel(OUT / "source-grid.pdf")
draw_target_panel(OUT / "target-density-grid.pdf")
copy_to_arxiv("source-grid.pdf")
copy_to_arxiv("target-density-grid.pdf")

In [5]:
# Thumbnail for the notebook gallery.
def render_pdf_to_image(pdf, scale=3.2):
    try:
        import fitz
        doc = fitz.open(pdf)
        page = doc[0]
        pix = page.get_pixmap(matrix=fitz.Matrix(scale, scale), alpha=False)
        im = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
        doc.close()
        return im
    except Exception:
        with tempfile.TemporaryDirectory() as tmp:
            prefix = Path(tmp) / "panel"
            subprocess.run(
                ["pdftoppm", "-png", "-r", str(int(72 * scale)), str(pdf), str(prefix)],
                check=True,
                stdout=subprocess.DEVNULL,
                stderr=subprocess.DEVNULL,
            )
            return Image.open(f"{prefix}-1.png").convert("RGB")

imgs = [render_pdf_to_image(OUT / "source-grid.pdf"), render_pdf_to_image(OUT / "target-density-grid.pdf")]
h = max(im.height for im in imgs)
pad = 14
canvas = Image.new("RGB", (sum(im.width for im in imgs) + 3 * pad, h + 2 * pad), "white")
x0 = pad
for im in imgs:
    canvas.paste(im, (x0, pad + (h - im.height) // 2))
    x0 += im.width + pad
thumb = THUMB_OUT / f"{NAME}.png"
canvas.save(thumb, quality=94)
print(thumb)


/Users/gpeyre/Dropbox/github/ot4ml/notebooks-figures/thumbnails/monge-jacobian-pushforward-density.png
